In [ ]:
!git clone https://github.com/MohammedMosuily/NLP_Lab5.git

Cloning into 'NLP_Lab5'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 3.29 MiB | 4.13 MiB/s, done.


In [ ]:
%cd NLP_Lab5
!ls -la

/content/NLP_Lab5/NLP_Lab5
total 72
drwxr-xr-x 4 root root  4096 Sep 28 15:45 .
drwxr-xr-x 5 root root  4096 Sep 28 15:45 ..
drwxr-xr-x 2 root root  4096 Sep 28 15:45 dataset
drwxr-xr-x 8 root root  4096 Sep 28 15:45 .git
-rw-r--r-- 1 root root 56692 Sep 28 15:45 Lab5_Text_Representation-2.ipynb


In [ ]:
import json

with open('Lab5_Text_Representation-2.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb.get('cells', [])):
    cell_type = cell.get('cell_type')
    source = ''.join(cell.get('source', []))
    print(f"=== Cell {i} ({cell_type}) ===")
    print(source[:500] + ('\n... [truncated]' if len(source) > 500 else ''))
    print("\n")

=== Cell 0 (markdown) ===
# Text Representation


=== Cell 1 (markdown) ===
Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.


=== Cell 2 (markdown) ===
## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.


=== Cell 3 (markdown) ===
<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">


=== Cell 4 (markdown) ===
# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">


=== Cell 5 (markdown) ===
TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieva

In [ ]:
import os
import pandas as pd

dataset_dir = 'dataset'
print("Files in dataset folder:", os.listdir(dataset_dir))

csv_path = os.path.join(dataset_dir, 'simpsons_script_lines.csv')
if os.path.exists(csv_path):
    df_simpsons = pd.read_csv(csv_path, low_memory=False)
    print("\nSimpsons columns:", df_simpsons.columns)
    print(df_simpsons[['spoken_words']].head())
else:
    print(f"\nError: {csv_path} not found.")

Files in dataset folder: ['simpsons_script_lines.csv']

Simpsons columns: Index(['raw_character_text', 'spoken_words'], dtype='object')
                                        spoken_words
0  No, actually, it was a little of both. Sometim...
1                             Where's Mr. Bergstrom?
2  I don't know. Although I'd sure like to talk t...
3                         That life is worth living.
4  The polls will be open from now until the end ...


In [ ]:
!pip install gensim

In [ ]:
import re
import pandas as pd
import gensim
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# Task 2: TF-IDF
# ==========================================
print("--- Task 2: TF-IDF ---")
task2_sentences = [
    'data science is one of the most important fields of science',
    'this is one of the best data science courses',
    'data scientists analyze data'
]
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(task2_sentences)
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out())
print(df_tfidf)
print("\n")

# ==========================================
# Task 3: Simpsons Preprocessing & Word2Vec
# ==========================================
print("--- Task 3: Simpsons Preprocessing & Word2Vec ---")
df_simpsons = pd.read_csv('dataset/simpsons_script_lines.csv', low_memory=False)
df_simpsons = df_simpsons.dropna(subset=['spoken_words'])

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

df_simpsons['cleaned_words'] = df_simpsons['spoken_words'].apply(clean_text)
tokens = [sentence.split() for sentence in df_simpsons['cleaned_words']]

Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
print("Word2Vec Model Trained Successfully!")
print("\n")

# ==========================================
# Task 4: Similarity Tests
# ==========================================
print("--- Task 4: Word Similarity ---")
for character in ["homer", "marge", "bart"]:
    if character in Skip_gram_model.wv:
        print(f"Words similar to '{character}':")
        print(Skip_gram_model.wv.most_similar(character, topn=5))
    else:
        print(f"'{character}' not found in vocabulary.")
print("\n")

# ==========================================
# Task 5: Odd One Out Tests
# ==========================================
print("--- Task 5: Odd One Out ---")
test_lists = [
    ['jimbo', 'milhouse', 'kearney'],
    ["nelson", "bart", "milhouse"],
    ['homer', 'patty', 'selma']
]

for lst in test_lists:
    valid_lst = [word for word in lst if word in Skip_gram_model.wv]
    if len(valid_lst) > 1:
        odd_one = Skip_gram_model.wv.doesnt_match(valid_lst)
        print(f"Odd one out among {valid_lst}: {odd_one}")
    else:
        print(f"Not enough words from {lst} are in vocabulary to test.")

--- Task 2: TF-IDF ---
    analyze      best   courses      data    fields  important        is  \
0  0.000000  0.000000  0.000000  0.189526  0.320895   0.320895  0.244049   
1  0.000000  0.400294  0.400294  0.236420  0.000000   0.000000  0.304434   
2  0.542701  0.000000  0.000000  0.641055  0.000000   0.000000  0.000000   

       most        of       one   science  scientists       the      this  
0  0.320895  0.488098  0.244049  0.488098    0.000000  0.244049  0.000000  
1  0.000000  0.304434  0.304434  0.304434    0.000000  0.304434  0.400294  
2  0.000000  0.000000  0.000000  0.000000    0.542701  0.000000  0.000000  


--- Task 3: Simpsons Preprocessing & Word2Vec ---


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Word2Vec Model Trained Successfully!


--- Task 4: Word Similarity ---
Words similar to 'homer':
[('marge', 0.8774037957191467), ('abe', 0.8757765293121338), ('bart', 0.8487610816955566), ('grampa', 0.8386150002479553), ('barney', 0.8353738188743591)]
Words similar to 'marge':
[('homer', 0.8774038553237915), ('abe', 0.8700903654098511), ('lisa', 0.8521142601966858), ('honey', 0.8437622785568237), ('bart', 0.8432631492614746)]
Words similar to 'bart':
[('lisa', 0.8969465494155884), ('milhouse', 0.8746446371078491), ('grampa', 0.8637741208076477), ('abe', 0.8589587807655334), ('apu', 0.8532668948173523)]


--- Task 5: Odd One Out ---
Odd one out among ['jimbo', 'milhouse', 'kearney']: milhouse
Odd one out among ['nelson', 'bart', 'milhouse']: nelson
Odd one out among ['homer', 'patty', 'selma']: homer


In [ ]:
import json

with open('Lab5_Text_Representation-2.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

# Insert solutions back into the respective blank code cells in the notebook file
# Cell 29: Task 2 Solution
nb['cells'][29]['source'] = [
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "import pandas as pd\n\n",
    "sentences = [\n",
    "    'data science is one of the most important fields of science',\n",
    "    'this is one of the best data science courses',\n",
    "    'data scientists analyze data'\n",
    "]\n\n",
    "tfidf_vec = TfidfVectorizer()\n",
    "tfidf_matrix = tfidf_vec.fit_transform(sentences)\n",
    "df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out())\n",
    "df_tfidf"
]

# Cell 32: Task 3 Solution (Clean & Train Skip-Gram Word2Vec)
nb['cells'][32]['source'] = [
    "import re\n",
    "import pandas as pd\n",
    "import gensim\n\n",
    "df_simpsons = pd.read_csv('dataset/simpsons_script_lines.csv', low_memory=False)\n",
    "df_simpsons = df_simpsons.dropna(subset=['spoken_words'])\n\n",
    "def clean_text(text):\n",
    "    text = text.lower()\n",
    "    text = re.sub(r\"[0-9]\", '', text)\n",
    "    text = re.sub(r\"[)(,”“.’$-]\", '', text)\n",
    "    return text\n\n",
    "df_simpsons['cleaned_words'] = df_simpsons['spoken_words'].apply(clean_text)\n",
    "tokens = [sentence.split() for sentence in df_simpsons['cleaned_words']]\n\n",
    "Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)\n",
    "print('Model Trained!')"
]

# Cell 34: Task 4 Solution
nb['cells'][34]['source'] = [
    "print('Homer similarities:', Skip_gram_model.wv.most_similar('homer'))\n",
    "print('Marge similarities:', Skip_gram_model.wv.most_similar('marge'))\n",
    "print('Bart similarities:', Skip_gram_model.wv.most_similar('bart'))"
]

# Cell 36: Task 5 Solution
nb['cells'][36]['source'] = [
    "print('Odd one out (jimbo, milhouse, kearney):', Skip_gram_model.wv.doesnt_match(['jimbo', 'milhouse', 'kearney']))\n",
    "print('Odd one out (nelson, bart, milhouse):', Skip_gram_model.wv.doesnt_match(['nelson', 'bart', 'milhouse']))\n",
    "print('Odd one out (homer, patty, selma):', Skip_gram_model.wv.doesnt_match(['homer', 'patty', 'selma']))"
]

# Save the completed notebook
with open('Lab5_Text_Representation-2_Completed.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=2)

print("Saved the completed notebook to Lab5_Text_Representation-2_Completed.ipynb")

Saved the completed notebook to Lab5_Text_Representation-2_Completed.ipynb
